# SAM-Med3D Segmentation Visualization

This notebook demonstrates proper SAM-Med3D segmentation using:
1. **Image encoder** for embeddings
2. **Prompt encoder + mask decoder** for actual segmentation
3. **Visualization** with proper contours

## Preprocessing
- ❌ NO ToCanonical
- ✅ YES CropOrPad(128³) - **ONLY THIS**
- ❌ NO ZNormalization

In [ ]:
import sys
from pathlib import Path
import numpy as np
import torch
import SimpleITK as sitk
import matplotlib.pyplot as plt
import random
import yaml
import torch.nn.functional as F

# Add project root to path
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from med3pipe.sam.core import build_sam3d_model, find_default_sam3d_root
import torchio as tio

## Configuration

In [ ]:
config_path = project_root / "configs" / "datasets.yaml"
with open(config_path) as f:
    config = yaml.safe_load(f)

sam3d_root = find_default_sam3d_root()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
img_size = 128
n_samples_per_dataset = 2

print(f"Using device: {device}")
print(f"SAM-Med3D root: {sam3d_root}")
print(f"Available datasets: {list(config['datasets'].keys())}")

## Build SAM-Med3D Model

In [ ]:
# Try to find checkpoint
checkpoint_path = sam3d_root / 'ckpt' / 'sam_med3d_turbo.pth'
if not checkpoint_path.exists():
    checkpoint_path = sam3d_root / 'ckpt' / 'SAM-Med3D-turbo.pth'
if not checkpoint_path.exists():
    print("⚠️  WARNING: No checkpoint found! Model will use random weights.")
    print(f"   Download from: https://huggingface.co/blueyo0/SAM-Med3D/blob/main/sam_med3d_turbo.pth")
    print(f"   Save to: {sam3d_root / 'ckpt' / 'sam_med3d_turbo.pth'}")
    checkpoint_path = None
else:
    print(f"✅ Using checkpoint: {checkpoint_path}")

model = build_sam3d_model(
    sam3d_root=sam3d_root,
    model_type="vit_b_ori",
    checkpoint=checkpoint_path,
    device=device,
    eval_mode=True
)
print("SAM-Med3D model loaded successfully!")

## Preprocessing Functions (ONLY CropOrPad)

In [ ]:
def load_volume_croponly(img_path, img_size=128):
    sitk_img = sitk.ReadImage(str(img_path))
    sitk_arr, _ = tio.data.io.sitk_to_nib(sitk_img)
    subject = tio.Subject(image=tio.ScalarImage(tensor=sitk_arr))
    crop_transform = tio.CropOrPad(target_shape=(img_size, img_size, img_size))
    subject = crop_transform(subject)
    image = subject.image.data.clone().detach()
    image = image.unsqueeze(0)
    image = image.float()  # Convert to float32
    return image

def load_mask_croponly(mask_path, img_size=128):
    sitk_mask = sitk.ReadImage(str(mask_path))
    mask_arr, _ = tio.data.io.sitk_to_nib(sitk_mask)
    subject = tio.Subject(label=tio.LabelMap(tensor=mask_arr))
    crop_transform = tio.CropOrPad(target_shape=(img_size, img_size, img_size))
    subject = crop_transform(subject)
    mask = subject.label.data.squeeze().numpy()
    mask = (mask > 0).astype(float)
    return mask

## Helper Functions

In [ ]:
def find_dataset_images(dataset_config, sam3d_root, project_root):
    images, labels = [], []
    category = dataset_config['category']
    ct_name = dataset_config['ct_name']
    
    possible_paths = [
        sam3d_root / 'data' / 'train' / category / ct_name / 'imagesTr',
        sam3d_root / 'data' / 'validation' / category / ct_name / 'imagesVal',
    ]
    
    for path in possible_paths:
        if path.exists():
            imgs = list(path.glob('*.nii.gz'))
            images.extend(imgs)
            label_path = path.parent / path.name.replace('images', 'labels')
            if label_path.exists():
                for img in imgs:
                    label_file = label_path / img.name
                    if label_file.exists():
                        labels.append(label_file)
    return images, labels

## SAM-Med3D Segmentation Functions

In [ ]:
def generate_sam_segmentation_proper(model, image_tensor, device, num_clicks=1):
    model.eval()
    with torch.no_grad():
        input_tensor = image_tensor.to(device)
        image_embeddings = model.image_encoder(input_tensor)
        
        D, H, W = input_tensor.shape[2:]
        center_point = torch.tensor([[[W//2, H//2, D//2]]], device=device, dtype=torch.float)
        point_labels = torch.tensor([[1]], device=device, dtype=torch.int64)
        low_res_shape = (1, 1, D//4, H//4, W//4)
        prev_low_res_mask = torch.zeros(low_res_shape, device=device, dtype=torch.float)
        
        for _ in range(num_clicks):
            sparse_embeddings, dense_embeddings = model.prompt_encoder(
                points=[center_point, point_labels], boxes=None, masks=prev_low_res_mask)
            low_res_masks, _ = model.mask_decoder(
                image_embeddings=image_embeddings,
                image_pe=model.prompt_encoder.get_dense_pe(),
                sparse_prompt_embeddings=sparse_embeddings,
                dense_prompt_embeddings=dense_embeddings,
                multimask_output=False)
            prev_low_res_mask = low_res_masks.detach()
        
        final_masks_hr = F.interpolate(low_res_masks, size=(D, H, W), mode='trilinear', align_corners=False)
        seg_prob = torch.sigmoid(final_masks_hr)
        seg_mask = (seg_prob > 0.5).cpu().squeeze().numpy().astype(np.uint8)
    return seg_mask

def generate_sam_with_bbox_prompt(model, image_tensor, gt_mask_tensor, device):
    model.eval()
    with torch.no_grad():
        input_tensor = image_tensor.to(device)
        image_embeddings = model.image_encoder(input_tensor)
        
        coords = np.argwhere(gt_mask_tensor > 0)
        if len(coords) == 0:
            return generate_sam_segmentation_proper(model, image_tensor, device)
        
        z_min, y_min, x_min = coords.min(axis=0)
        z_max, y_max, x_max = coords.max(axis=0)
        z_margin = max(1, int((z_max - z_min) * 0.1))
        y_margin = max(1, int((y_max - y_min) * 0.1))
        x_margin = max(1, int((x_max - x_min) * 0.1))
        
        D, H, W = input_tensor.shape[2:]
        z_min = max(0, z_min - z_margin)
        z_max = min(D-1, z_max + z_margin)
        y_min = max(0, y_min - y_margin)
        y_max = min(H-1, y_max + y_margin)
        x_min = max(0, x_min - x_margin)
        x_max = min(W-1, x_max + x_margin)
        
        center_x = (x_min + x_max) // 2
        center_y = (y_min + y_max) // 2
        center_z = (z_min + z_max) // 2
        
        points_list = [[center_x, center_y, center_z],
                      [x_min + (x_max-x_min)//4, center_y, center_z],
                      [x_max - (x_max-x_min)//4, center_y, center_z]]
        
        points_coords = torch.tensor([points_list], device=device, dtype=torch.float)
        point_labels = torch.tensor([[1, 1, 1]], device=device, dtype=torch.int64)
        low_res_shape = (1, 1, D//4, H//4, W//4)
        prev_low_res_mask = torch.zeros(low_res_shape, device=device, dtype=torch.float)
        
        sparse_embeddings, dense_embeddings = model.prompt_encoder(
            points=[points_coords, point_labels], boxes=None, masks=prev_low_res_mask)
        low_res_masks, _ = model.mask_decoder(
            image_embeddings=image_embeddings,
            image_pe=model.prompt_encoder.get_dense_pe(),
            sparse_prompt_embeddings=sparse_embeddings,
            dense_prompt_embeddings=dense_embeddings,
            multimask_output=False)
        
        final_masks_hr = F.interpolate(low_res_masks, size=(D, H, W), mode='trilinear', align_corners=False)
        seg_prob = torch.sigmoid(final_masks_hr)
        seg_mask = (seg_prob > 0.5).cpu().squeeze().numpy().astype(np.uint8)
    return seg_mask

## Plotting Function

In [ ]:
def plot_slices(image_vol, seg_vol, gt_mask_vol=None, title="", n_slices=5):
    D, H, W = image_vol.shape
    slice_indices = np.linspace(D//4, 3*D//4, n_slices, dtype=int)
    n_cols = 3 if gt_mask_vol is not None else 2
    fig, axes = plt.subplots(n_slices, n_cols, figsize=(4*n_cols, 3*n_slices))
    
    if n_slices == 1:
        axes = axes.reshape(1, -1)
    
    for i, slice_idx in enumerate(slice_indices):
        axes[i, 0].imshow(image_vol[slice_idx], cmap='gray')
        axes[i, 0].set_title(f"Original (slice {slice_idx})")
        axes[i, 0].axis('off')
        
        axes[i, 1].imshow(image_vol[slice_idx], cmap='gray')
        seg_overlay = np.ma.masked_where(seg_vol[slice_idx] == 0, seg_vol[slice_idx])
        axes[i, 1].imshow(seg_overlay, cmap='Reds', alpha=0.5, vmin=0, vmax=1)
        axes[i, 1].set_title(f"SAM Segmentation (slice {slice_idx})")
        axes[i, 1].axis('off')
        
        if gt_mask_vol is not None:
            axes[i, 2].imshow(image_vol[slice_idx], cmap='gray')
            gt_overlay = np.ma.masked_where(gt_mask_vol[slice_idx] == 0, gt_mask_vol[slice_idx])
            axes[i, 2].imshow(gt_overlay, cmap='Greens', alpha=0.5, vmin=0, vmax=1)
            axes[i, 2].set_title(f"Ground Truth (slice {slice_idx})")
            axes[i, 2].axis('off')
    
    fig.suptitle(title, fontsize=16, y=0.995)
    plt.tight_layout()
    return fig

## Process Datasets

In [ ]:
random.seed(42)
np.random.seed(42)

print("📋 Preprocessing: ONLY CropOrPad(128³)")
print("   ❌ ToCanonical disabled")
print("   ✅ CropOrPad(128³) enabled")
print("   ❌ ZNormalization disabled\n")

for dataset_name, dataset_config in config['datasets'].items():
    print(f"\n{'='*60}")
    print(f"Processing dataset: {dataset_name.upper()}")
    print(f"{'='*60}")
    
    images, labels = find_dataset_images(dataset_config, sam3d_root, project_root)
    
    if not images:
        print(f"⚠️  No images found for {dataset_name}. Skipping...")
        continue
    
    print(f"Found {len(images)} images")
    n_samples = min(n_samples_per_dataset, len(images))
    selected_indices = random.sample(range(len(images)), n_samples)
    
    for idx in selected_indices:
        img_path = images[idx]
        print(f"\nProcessing: {img_path.name}")
        
        try:
            print(f"  Loading image (CropOrPad only)...")
            image_tensor = load_volume_croponly(img_path, img_size=img_size)
            print(f"    Image shape: {image_tensor.shape}")
            
            gt_mask = None
            if idx < len(labels) and labels[idx].exists():
                print("  Loading ground truth mask...")
                gt_mask = load_mask_croponly(labels[idx], img_size=img_size)
                print(f"    Mask shape: {gt_mask.shape}")
            
            print("  Generating SAM segmentation (with proper decoder)...")
            if gt_mask is not None:
                sam_seg = generate_sam_with_bbox_prompt(model, image_tensor, gt_mask, device)
            else:
                sam_seg = generate_sam_segmentation_proper(model, image_tensor, device)
            
            image_np = image_tensor.squeeze().cpu().numpy()
            
            print("  Plotting...")
            fig = plot_slices(image_np, sam_seg, gt_mask,
                            title=f"{dataset_name.upper()} - {img_path.stem}", n_slices=5)
            plt.show()
            
        except Exception as e:
            print(f"  ❌ Error processing {img_path.name}: {e}")
            import traceback
            traceback.print_exc()
            continue

print("\n" + "="*60)
print("Processing complete!")
print("="*60)

## Test with Toy Data

In [ ]:
test_data_path = sam3d_root / 'test_data' / 'amos_val_toy_data'

if test_data_path.exists():
    print(f"{'='*60}")
    print("Testing with toy data")
    print(f"{'='*60}")
    
    images_path = test_data_path / 'imagesVa'
    labels_path = test_data_path / 'labelsVa'
    test_images = list(images_path.glob('*.nii.gz'))
    
    if test_images:
        print(f"Found {len(test_images)} test images")
        
        for img_path in test_images[:1]:
            print(f"\nProcessing test image: {img_path.name}")
            
            try:
                print(f"  Loading image (CropOrPad only)...")
                image_tensor = load_volume_croponly(img_path, img_size=img_size)
                print(f"    Image shape: {image_tensor.shape}")
                
                gt_mask = None
                label_path = labels_path / img_path.name
                if label_path.exists():
                    print("  Loading ground truth mask...")
                    gt_mask = load_mask_croponly(label_path, img_size=img_size)
                    print(f"    Mask shape: {gt_mask.shape}")
                
                print("  Generating SAM segmentation...")
                if gt_mask is not None:
                    sam_seg = generate_sam_with_bbox_prompt(model, image_tensor, gt_mask, device)
                else:
                    sam_seg = generate_sam_segmentation_proper(model, image_tensor, device)
                
                image_np = image_tensor.squeeze().cpu().numpy()
                
                print("  Plotting...")
                fig = plot_slices(image_np, sam_seg, gt_mask,
                                title=f"Test Data - {img_path.stem}", n_slices=5)
                plt.show()
                
            except Exception as e:
                print(f"  ❌ Error: {e}")
                import traceback
                traceback.print_exc()
else:
    print("\nNo test data found.")